# Prior Sensitivity Analysis in Bayesian Commodity Forecasting

## Learning Objectives

By completing this notebook, you will:
1. Understand why prior sensitivity analysis is crucial for credible Bayesian inference
2. Quantify how posterior conclusions depend on prior specifications
3. Use multiple priors to assess robustness of commodity price forecasts
4. Apply prior predictive checks to detect unrealistic priors
5. Use weakly informative priors that regularize without dominating data
6. Report sensitivity analyses professionally in applied work

## Prerequisites
- Bayesian inference fundamentals
- PyMC model building
- Understanding of conjugate priors

## Estimated Time: 90 minutes

---

## 1. Setup and Imports

We'll use PyMC for Bayesian modeling and ArviZ for diagnostics.

In [ ]:
# Core imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Bayesian stack
import pymc as pm
import arviz as az
from scipy import stats

# Configuration
np.random.seed(42)
az.style.use('arviz-darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)

print(f"PyMC version: {pm.__version__}")
print(f"ArviZ version: {az.__version__}")
print("Environment ready!")

## 2. Why Prior Sensitivity Matters

**The Problem:**
- Priors inject subjective information into analysis
- Different analysts might choose different priors
- If conclusions depend heavily on priors, they're not data-driven

**The Goal:**
- Show that conclusions are **robust** to reasonable prior choices
- Identify when priors dominate (small data) vs when data dominates (large data)
- Build trust in Bayesian results

**Best Practice:**
Always report results under multiple prior specifications. If conclusions flip with reasonable prior changes, you don't have enough data yet.

**Three Types of Priors:**
1. **Flat/Uninformative**: Minimal prior knowledge (e.g., Uniform, very wide Normal)
2. **Weakly Informative**: Gentle regularization (e.g., Normal(0, 10) for standardized coefficients)
3. **Informative**: Strong domain knowledge (e.g., Normal(-5, 1) if we're confident inventory lowers prices)

## 3. Commodity Price Example: Natural Gas Storage

**Scenario:** Model natural gas prices as a function of storage levels.

**Domain Knowledge:**
- Higher storage → lower prices (basic supply/demand)
- Relationship is approximately linear in log space
- But the exact coefficient is uncertain

We'll fit:
$$\text{Price}_i = \alpha + \beta \times \text{Storage}_i + \epsilon_i$$

with different priors on $\beta$ and see how results change.

In [ ]:
# Generate synthetic natural gas data
np.random.seed(42)

# True parameters (unknown to our models)
true_alpha = 5.0  # Base price level
true_beta = -0.8  # Storage effect (negative: more storage → lower price)
true_sigma = 0.5  # Noise

# Generate data
n_obs = 30  # Limited data (common in commodity markets)
storage = np.random.normal(0, 1, n_obs)  # Standardized storage levels
price = true_alpha + true_beta * storage + np.random.normal(0, true_sigma, n_obs)

# Create DataFrame
data = pd.DataFrame({'storage': storage, 'price': price})

print(f"Data: {n_obs} observations")
print(f"True beta (storage effect): {true_beta}")
print(f"\nData summary:")
print(data.describe())

In [ ]:
# Visualize relationship
fig, ax = plt.subplots(figsize=(10, 6))

ax.scatter(data['storage'], data['price'], s=80, alpha=0.6, edgecolors='black')
ax.set_xlabel('Storage Level (standardized)', fontsize=12)
ax.set_ylabel('Natural Gas Price', fontsize=12)
ax.set_title('Natural Gas Price vs Storage Level', fontsize=14)
ax.grid(True, alpha=0.3)

# Add OLS line for reference
from scipy.stats import linregress
slope, intercept, r, p, se = linregress(data['storage'], data['price'])
x_line = np.linspace(data['storage'].min(), data['storage'].max(), 100)
ax.plot(x_line, intercept + slope * x_line, 'r--', linewidth=2, 
        label=f'OLS: β={slope:.2f}', alpha=0.7)
ax.legend()

plt.tight_layout()
plt.show()

print(f"OLS estimate for beta: {slope:.3f} ± {1.96*se:.3f}")

## 4. Model Setup: Three Different Priors

We'll fit the same model with three different prior specifications for $\beta$:

**Prior Set 1: Flat (Uninformative)**
- $\beta \sim \mathcal{N}(0, 100^2)$ — Very wide, almost uniform

**Prior Set 2: Weakly Informative**
- $\beta \sim \mathcal{N}(0, 2^2)$ — Expects coefficient around 0, but allows variation

**Prior Set 3: Informative (Domain Knowledge)**
- $\beta \sim \mathcal{N}(-1, 0.5^2)$ — Encodes belief that storage reduces prices

All three use the same priors for $\alpha$ and $\sigma$.

In [ ]:
# Define prior specifications
prior_specs = {
    'Flat': {'mu': 0, 'sigma': 100},
    'Weakly Informative': {'mu': 0, 'sigma': 2},
    'Informative': {'mu': -1, 'sigma': 0.5}
}

# Visualize priors
beta_grid = np.linspace(-3, 1, 1000)

fig, ax = plt.subplots(figsize=(12, 6))

for name, params in prior_specs.items():
    prior_pdf = stats.norm(params['mu'], params['sigma']).pdf(beta_grid)
    ax.plot(beta_grid, prior_pdf, linewidth=2, label=name)

ax.axvline(true_beta, color='black', linestyle='--', linewidth=2, 
           label=f'True β: {true_beta}')
ax.set_xlabel('β (Storage Effect)', fontsize=12)
ax.set_ylabel('Prior Density', fontsize=12)
ax.set_title('Three Different Prior Specifications for β', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Fit Models with Different Priors

Now we'll fit all three models and store results for comparison.

In [ ]:
# Dictionary to store traces
traces = {}

# Fit each model
for prior_name, prior_params in prior_specs.items():
    print(f"\nFitting model with {prior_name} prior...")
    
    with pm.Model() as model:
        # Priors
        alpha = pm.Normal('alpha', mu=5, sigma=2)
        beta = pm.Normal('beta', mu=prior_params['mu'], sigma=prior_params['sigma'])
        sigma = pm.HalfNormal('sigma', sigma=1)
        
        # Likelihood
        mu = alpha + beta * data['storage']
        y_obs = pm.Normal('y_obs', mu=mu, sigma=sigma, observed=data['price'])
        
        # Sample
        trace = pm.sample(
            draws=2000,
            tune=1000,
            chains=2,
            cores=1,
            random_seed=42,
            return_inferencedata=True
        )
        
        traces[prior_name] = trace
    
    print(f"✓ {prior_name} model complete")

print("\n✅ All models fitted!")

## 6. Compare Posteriors Across Priors

Let's see how the posterior for $\beta$ (storage effect) changes with different priors.

In [ ]:
# Extract posterior summaries
results = []

for prior_name, trace in traces.items():
    summary = az.summary(trace, var_names=['beta'])
    results.append({
        'Prior': prior_name,
        'Posterior Mean': summary['mean'].values[0],
        'Posterior Std': summary['sd'].values[0],
        'HDI 2.5%': summary['hdi_3%'].values[0],
        'HDI 97.5%': summary['hdi_97%'].values[0]
    })

comparison_df = pd.DataFrame(results)
print("Posterior Comparison for β (Storage Effect):")
print("="*70)
print(comparison_df.to_string(index=False))
print("="*70)
print(f"True β: {true_beta}")
print(f"OLS estimate: {slope:.3f}")

In [ ]:
# Visualize posterior distributions
fig, ax = plt.subplots(figsize=(12, 6))

colors = ['blue', 'green', 'red']
for (prior_name, trace), color in zip(traces.items(), colors):
    beta_samples = trace.posterior['beta'].values.flatten()
    ax.hist(beta_samples, bins=50, density=True, alpha=0.4, 
            color=color, label=f'{prior_name} Posterior')
    
    # Add mean line
    mean_beta = beta_samples.mean()
    ax.axvline(mean_beta, color=color, linestyle='--', linewidth=2, alpha=0.7)

# True value
ax.axvline(true_beta, color='black', linestyle='--', linewidth=3, 
           label=f'True β: {true_beta}')

ax.set_xlabel('β (Storage Effect)', fontsize=12)
ax.set_ylabel('Posterior Density', fontsize=12)
ax.set_title('Posterior Distributions Under Different Priors', fontsize=14)
ax.legend(fontsize=10, loc='upper left')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

**Observations:**

1. **Flat prior**: Posterior nearly identical to OLS (data dominates)
2. **Weakly informative**: Slight regularization toward 0, but data still dominates
3. **Informative**: Posterior is pulled toward prior mean (-1), creating a compromise

**Key Question:** Are our conclusions robust?
- If we want to claim "storage reduces prices," all three posteriors support this (all negative)
- If we need precise magnitude, there's more disagreement

## 7. Prior vs Posterior Weight

Quantify how much the data moved us from the prior.

**Prior-Posterior Shift:**
$$\text{Shift} = |\text{Posterior Mean} - \text{Prior Mean}|$$

**Prior Weight:**
$$\text{Prior Weight} \approx \frac{\text{Posterior Std}^2}{\text{Prior Std}^2}$$

Smaller ratio → data had more influence.

In [ ]:
# Compute prior-posterior metrics
metrics = []

for prior_name, prior_params in prior_specs.items():
    trace = traces[prior_name]
    
    # Prior stats
    prior_mean = prior_params['mu']
    prior_std = prior_params['sigma']
    
    # Posterior stats
    beta_samples = trace.posterior['beta'].values.flatten()
    post_mean = beta_samples.mean()
    post_std = beta_samples.std()
    
    # Metrics
    shift = abs(post_mean - prior_mean)
    uncertainty_reduction = 1 - (post_std / prior_std)
    
    metrics.append({
        'Prior': prior_name,
        'Prior Mean': prior_mean,
        'Posterior Mean': post_mean,
        'Shift': shift,
        'Prior Std': prior_std,
        'Posterior Std': post_std,
        'Uncertainty Reduction': uncertainty_reduction
    })

metrics_df = pd.DataFrame(metrics)
print("Prior vs Posterior Comparison:")
print("="*90)
print(metrics_df.to_string(index=False))
print("="*90)

**Interpretation:**

- **Flat prior**: Large shift (data dominates), large uncertainty reduction
- **Weakly informative**: Moderate shift, good uncertainty reduction
- **Informative**: Small shift (prior + data compromise), less uncertainty reduction

## 8. Prior Predictive Checks

**Before looking at data**, check if priors produce realistic predictions.

**Method:**
1. Sample parameters from prior
2. Generate fake data using those parameters
3. Check if fake data is plausible

**Red flags:**
- Negative prices
- Prices outside realistic range
- Relationships that violate physics/economics

In [ ]:
# Prior predictive check for each prior
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (prior_name, prior_params) in zip(axes, prior_specs.items()):
    # Build model
    with pm.Model() as model:
        alpha = pm.Normal('alpha', mu=5, sigma=2)
        beta = pm.Normal('beta', mu=prior_params['mu'], sigma=prior_params['sigma'])
        sigma = pm.HalfNormal('sigma', sigma=1)
        
        mu = alpha + beta * data['storage']
        y_obs = pm.Normal('y_obs', mu=mu, sigma=sigma, observed=data['price'])
        
        # Sample prior predictive
        prior_pred = pm.sample_prior_predictive(samples=500, random_seed=42)
    
    # Extract predictions
    prior_samples = prior_pred.prior_predictive['y_obs'].values
    
    # Plot
    for i in range(50):  # Plot 50 random draws
        ax.plot(data['storage'], prior_samples[0, i, :], 'b-', alpha=0.05)
    
    ax.scatter(data['storage'], data['price'], color='red', s=30, 
               zorder=10, label='Actual Data', alpha=0.7)
    ax.set_xlabel('Storage', fontsize=11)
    ax.set_ylabel('Price', fontsize=11)
    ax.set_title(f'{prior_name} Prior\nPredictive Check', fontsize=12)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(-5, 15)

plt.tight_layout()
plt.show()

**Assessment:**

- **Flat prior**: Blue lines spread too wide, many unrealistic prices (negative, very high)
- **Weakly informative**: Reasonable spread, mostly plausible
- **Informative**: Tight spread around negative slope, very constrained

**Verdict:** Weakly informative strikes best balance.

## 9. Posterior Predictive Checks

After fitting, check if models can **reproduce** the observed data.

In [ ]:
# Posterior predictive check
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (prior_name, trace) in zip(axes, traces.items()):
    # Generate posterior predictive samples
    with pm.Model() as model:
        alpha = pm.Normal('alpha', mu=5, sigma=2)
        beta = pm.Normal('beta', mu=prior_specs[prior_name]['mu'], 
                        sigma=prior_specs[prior_name]['sigma'])
        sigma = pm.HalfNormal('sigma', sigma=1)
        
        mu = alpha + beta * data['storage']
        y_obs = pm.Normal('y_obs', mu=mu, sigma=sigma, observed=data['price'])
        
        post_pred = pm.sample_posterior_predictive(trace, random_seed=42)
    
    # Plot
    az.plot_ppc(post_pred, ax=ax, observed_rug=True, num_pp_samples=100)
    ax.set_title(f'{prior_name} Prior\nPosterior Predictive Check', fontsize=12)
    ax.set_xlabel('Price', fontsize=11)

plt.tight_layout()
plt.show()

**Interpretation:**
- Blue lines: Simulated data from posterior
- Black line: Observed data
- Good fit: Blue and black align

All three models fit the data reasonably well (posterior predictive matches observed).

## 10. Decision-Making Under Prior Uncertainty

**Scenario:** We need to decide if storage has a "meaningful" negative effect on price.

**Decision Rule:** If $\beta < -0.5$ with >80% probability, we'll adjust trading strategy.

Let's see how this probability changes with priors.

In [ ]:
# Compute decision probabilities
threshold = -0.5

decision_results = []

for prior_name, trace in traces.items():
    beta_samples = trace.posterior['beta'].values.flatten()
    
    # Probability beta < threshold
    prob_below = (beta_samples < threshold).mean()
    
    # Decision
    decision = "ADJUST STRATEGY" if prob_below > 0.80 else "NO ACTION"
    
    decision_results.append({
        'Prior': prior_name,
        f'P(β < {threshold})': prob_below,
        'Decision': decision
    })

decision_df = pd.DataFrame(decision_results)
print(f"Decision Analysis: P(β < {threshold})")
print("="*60)
print(decision_df.to_string(index=False))
print("="*60)

**Key Insight:**

If our decision depends critically on prior choice, we need:
1. More data to reduce prior influence
2. Stronger justification for chosen prior
3. Transparency about prior sensitivity in reporting

---

## Exercise 1: Sensitivity to Sample Size

**Task:** Repeat the analysis with different sample sizes (n = 10, 30, 100) using the **Informative** prior.

**Questions:**
1. At what sample size does the posterior stop depending on the prior?
2. Plot how the posterior mean changes with sample size

In [ ]:
# YOUR CODE HERE
np.random.seed(42)

sample_sizes = [10, 30, 100]
results_ex1 = []

# Use informative prior: N(-1, 0.5)
prior_mu_ex1 = -1
prior_sigma_ex1 = 0.5

for n in sample_sizes:
    # Generate data
    storage_ex1 = np.random.normal(0, 1, n)
    price_ex1 = true_alpha + true_beta * storage_ex1 + np.random.normal(0, true_sigma, n)
    
    # Fit model
    # YOUR CODE: Build and fit PyMC model with informative prior
    # Store posterior mean and std for beta
    
    with pm.Model() as model_ex1:
        # YOUR CODE HERE
        pass
    
    # Extract results
    # results_ex1.append({'n': n, 'post_mean': ..., 'post_std': ...})

# Plot results
# YOUR CODE: Create plot showing how posterior mean converges to true value

In [ ]:
# AUTO-GRADED TEST
def test_exercise_1():
    """Check that sample size analysis was performed."""
    assert len(results_ex1) == len(sample_sizes), (
        f"Expected {len(sample_sizes)} results, got {len(results_ex1)}"
    )
    
    # Check that posterior means exist
    for result in results_ex1:
        assert 'post_mean' in result, "Missing 'post_mean' in results"
        assert result['post_mean'] is not None, "post_mean is None"
    
    # Check that uncertainty decreased with sample size
    stds = [r['post_std'] for r in results_ex1]
    assert stds[0] > stds[-1], (
        "Posterior std should decrease with sample size. "
        f"Got: n={sample_sizes[0]} std={stds[0]:.3f}, "
        f"n={sample_sizes[-1]} std={stds[-1]:.3f}"
    )
    
    print("✅ Exercise 1 passed!")

# test_exercise_1()  # Uncomment after completing exercise

---

## Exercise 2: Prior Predictive Rejection

**Task:** Create a prior that produces unrealistic predictions (e.g., mostly negative prices).

1. Choose prior parameters that fail the prior predictive check
2. Fit the model anyway
3. Show that even bad priors can be overcome by enough data

In [ ]:
# YOUR CODE HERE

# Step 1: Define a bad prior (hint: large positive beta → prices increase with storage, unrealistic)
bad_prior_mu = None  # <- Choose a value
bad_prior_sigma = None  # <- Choose a value

# Step 2: Run prior predictive check
# YOUR CODE: Show that prior predictive gives unrealistic prices

# Step 3: Fit model anyway
# YOUR CODE: Fit model with bad prior

# Step 4: Show posterior is still reasonable
# YOUR CODE: Compare prior mean vs posterior mean

In [ ]:
# AUTO-GRADED TEST
def test_exercise_2():
    """Verify bad prior was chosen and model still recovered."""
    assert bad_prior_mu is not None, "bad_prior_mu is None"
    assert bad_prior_sigma is not None, "bad_prior_sigma is None"
    
    # Check that prior is indeed "bad" (far from true value)
    assert abs(bad_prior_mu - true_beta) > 2, (
        f"Prior should be far from true value {true_beta}. "
        f"Got prior mean {bad_prior_mu}"
    )
    
    print("✅ Exercise 2 setup complete!")
    print("Make sure to show that the posterior still recovers the true value.")

# test_exercise_2()  # Uncomment after completing exercise

---

## Exercise 3: Robustness Report

**Task:** Write a professional sensitivity analysis report for a commodity trading desk.

**Requirements:**
1. State the research question clearly
2. Describe all priors tested and their justifications
3. Present posterior comparison table
4. Make a recommendation with uncertainty quantification
5. Discuss when results would be trustworthy vs when more data is needed

### YOUR REPORT HERE

**Title:** Prior Sensitivity Analysis: Natural Gas Storage Effect

**Research Question:**
[Your text here]

**Priors Tested:**
[Your text here]

**Results:**
[Your text here]

**Recommendation:**
[Your text here]

**Limitations:**
[Your text here]

---

## Exercise 4: Multiple Parameters

**Task:** Extend to a multivariate sensitivity analysis.

Add a second predictor (e.g., `production`) to the model:
$$\text{Price}_i = \alpha + \beta_1 \times \text{Storage}_i + \beta_2 \times \text{Production}_i + \epsilon_i$$

Test sensitivity of **both** $\beta_1$ and $\beta_2$ to different priors.

**Challenge:** Show how correlation between predictors affects prior sensitivity.

In [ ]:
# YOUR CODE HERE

# Generate data with two predictors
np.random.seed(42)
production = np.random.normal(0, 1, n_obs)

# True model: Price = 5 - 0.8*Storage + 0.6*Production + noise
true_beta1 = -0.8
true_beta2 = 0.6

price_multi = (true_alpha + true_beta1 * storage + true_beta2 * production + 
               np.random.normal(0, true_sigma, n_obs))

# YOUR CODE: Test different prior combinations for beta1 and beta2

---

## Solutions

### Exercise 1 Solution (Abbreviated)

In [ ]:
# SOLUTION
# for n in sample_sizes:
#     storage_ex1 = np.random.normal(0, 1, n)
#     price_ex1 = true_alpha + true_beta * storage_ex1 + np.random.normal(0, true_sigma, n)
#     
#     with pm.Model() as model_ex1:
#         alpha = pm.Normal('alpha', mu=5, sigma=2)
#         beta = pm.Normal('beta', mu=-1, sigma=0.5)  # Informative
#         sigma = pm.HalfNormal('sigma', sigma=1)
#         mu = alpha + beta * storage_ex1
#         y_obs = pm.Normal('y_obs', mu=mu, sigma=sigma, observed=price_ex1)
#         trace_ex1 = pm.sample(1000, tune=500, chains=2, progressbar=False)
#     
#     beta_samples = trace_ex1.posterior['beta'].values.flatten()
#     results_ex1.append({
#         'n': n,
#         'post_mean': beta_samples.mean(),
#         'post_std': beta_samples.std()
#     })
#
# Key finding: With n=100, prior influence is minimal; posterior ≈ true_beta

---

## Summary

### Key Takeaways

1. **Always test multiple priors** to assess robustness of conclusions
2. **Prior predictive checks** catch unrealistic priors before fitting
3. **Weakly informative priors** balance regularization and data influence
4. **With enough data**, even bad priors are overcome
5. **Report sensitivity**: Transparency builds trust in Bayesian analyses
6. **Small data = high prior influence**: Be extra careful with prior choice

### Best Practices for Applied Work

1. **Default to weakly informative priors** (e.g., Normal(0, 2) for standardized data)
2. **Justify informative priors** with domain expertise or previous studies
3. **Always include sensitivity analysis** in reports
4. **Use prior predictive checks** to validate prior realism
5. **Report HDI intervals**, not just point estimates

### What's Next

Module 2: **Commodity Data Pipelines** - Where does the data come from?

### Additional Resources

- Gelman et al. (2017): "Prior Choice Recommendations"
- Gabry et al. (2019): "Visualization in Bayesian workflow"
- Stan Prior Choice Wiki: https://github.com/stan-dev/stan/wiki/Prior-Choice-Recommendations

---

*Remember: Bayesian inference is transparent about subjective choices. Prior sensitivity analysis is how we prove our conclusions are data-driven, not prior-driven.*